# Etapa 4: Resultados — Salario mínimo y desempleo en Chile (jóvenes vs. adultos)

**Estructura del notebook**

| Sección | Contenido |
|---|---|
| 0 | Librerías, configuración y preparación de `df` |
| 1 | Modelo principal con interacción (4.1) |
| 2 | Regresiones separadas por grupo (4.2) |
| 3 | Supuestos: Jarque-Bera, VIF, Breusch-Godfrey, White, ADF y reestimación HAC (4.3) |
| 4 | Robustez: ocupación, salario real, rezagos, sin COVID, tendencia lineal (4.4) |
| 5 | Tabla de coeficientes con interpretación, tabla de resultados de las regresiones y conclusión sobre H1 y H2 (4.5) |

**Lectura de los coeficientes (modelo nivel-log):** la dependiente está en % y el Kaitz en logaritmo, así que
$\beta/100$ = puntos porcentuales (pp) de desempleo ante un alza de **1 %** del Kaitz, y $\beta \times \ln(1{,}10) \approx \beta/10$ = pp ante un alza de **10 %**.

- $\beta_1$: efecto en adultos · $\beta_1+\beta_3$: efecto en jóvenes · $\beta_3$: diferencia (prueba H2).

**Datos:** `datos_procesados/base_apilada.csv` (generada por `construccion_variables.ipynb`): cada trimestre aparece dos veces (jóvenes y adultos), 67 trimestres por grupo. Este archivo debe estar en la carpeta raíz del repositorio, junto a `construccion_variables.ipynb`.

## Mapa de los modelos que se estiman

En total se estiman **9 modelos**: el principal, dos regresiones por grupo y seis especificaciones de robustez. Todas son **OLS** (mínimos cuadrados ordinarios) con datos trimestrales 2010-T1 a 2026-T3.

| # | Modelo | Sección | Variable dependiente | Variable de salario mínimo | Muestra | Pregunta que responde |
|---|---|---|---|---|---|---|
| 1 | **Principal con interacción** | 1 y 3.6 | Desempleo | ln Kaitz (centrado) | Ambos grupos, N = 134 | ¿El Kaitz aumenta el desempleo (H1)? ¿Más en jóvenes (H2)? |
| 2 | Solo jóvenes | 2 | Desempleo | ln Kaitz | Jóvenes, N = 67 | ¿Cuál es el efecto en jóvenes, estimado por separado? |
| 3 | Solo adultos | 2 | Desempleo | ln Kaitz | Adultos, N = 67 | ¿Cuál es el efecto en adultos, estimado por separado? |
| 4 | Ocupación | 4.1 | **Tasa de ocupación** | ln Kaitz | N = 134 | ¿Se mantiene el resultado con la medida que prefiere la literatura? |
| 5 | Salario real | 4.2 | Desempleo | **ln salario mínimo real** | N = 134 | ¿Depende el resultado de usar el Kaitz? |
| 6 | Rezago 1 | 4.3 | Desempleo | **ln Kaitz del trimestre anterior** | N = 132 | ¿El efecto aparece con retraso? |
| 7 | Rezago 2 | 4.3 | Desempleo | **ln Kaitz de hace 2 trimestres** | N = 130 | Ídem, con 6 meses de retraso |
| 8 | Sin COVID | 4.4 | Desempleo | ln Kaitz | **Sin 2020-T2 a 2021-T2**, N = 124 | ¿La pandemia determina el resultado? |
| 9 | Con tendencia | 4.5 | Desempleo | ln Kaitz | N = 134 | ¿El Kaitz solo está capturando una tendencia temporal? |

### Variables que aparecen en los modelos

| Variable en el código | Columna original | Qué mide | Unidad | Rol |
|---|---|---|---|---|
| `desempleo` | `u` | Tasa de desocupación del grupo | % | Dependiente principal |
| `tasa_ocupacion` | `o` | Ocupados / población en edad de trabajar | % | Dependiente (modelo 4) |
| `ln_Kaitz_c` | `ln_kaitz` | Log del Kaitz (salario mínimo / salario promedio), centrado en su media | log | **Explicativa principal** |
| `D_joven` | `d_joven` | 1 = 15–24 años, 0 = 25 años o más | 0/1 | Grupo |
| `interaccion_joven` | — | `ln_Kaitz_c` × `D_joven` | log | **Prueba de H2** |
| `ln_salario_real_c` | `ln_sm_real` | Log del salario mínimo real (pesos 2023), centrado | log | Explicativa (modelo 5) |
| `IMACEC` | `imacec_va` | Variación anual del Imacec | % | Control del ciclo |
| `inflacion` | `inflacion` | Variación del IPC en 12 meses | % | Control |
| `COVID` | `covid` | 1 entre 2020-T2 y 2021-T2 | 0/1 | Control |
| `Q1`, `Q2`, `Q3` | `q1`–`q3` | Trimestre ene–mar, abr–jun, jul–sep (base: oct–dic) | 0/1 | Control de estacionalidad |
| `tendencia` | `tendencia` | 1, 2, …, 67 | trimestres | Control (modelo 9) |

### Reglas generales de interpretación
La dependiente está en **%**, así que todos los coeficientes se leen en **puntos porcentuales (pp)**:

| Tipo de variable | Lectura del coeficiente β |
|---|---|
| En logaritmo (Kaitz, salario real) | Modelo **nivel-log**: un alza de 1 % en la variable cambia la dependiente en β/100 pp; un alza de 10 %, en β × ln(1,10) ≈ β/10 pp |
| En % (Imacec, inflación) | Un aumento de 1 punto en la variable cambia la dependiente en β pp |
| Dummy (joven, COVID, trimestres) | Diferencia en pp respecto del grupo base (adultos, sin COVID, oct–dic) |
| Tendencia | Cambio en pp por cada trimestre que pasa |

---
## 0. Preparación

### 0.0 Librerías y parámetros
- **`statsmodels.formula.api` (`smf`)** permite escribir los modelos como fórmulas al estilo de R (`y ~ x1 + x2`) y agrega el intercepto automáticamente.
- Del módulo `statsmodels.stats` se importan los tests de supuestos y de `statsmodels.tsa` el test ADF.
- **`ALPHA = 0.05`**: nivel de significancia que usan todos los tests para decidir si se rechaza la hipótesis nula.
- **`LAGS_BG = 4`**: el test de Breusch-Godfrey revisa si el error de un trimestre se relaciona con el de los 4 trimestres anteriores, es decir, un año completo.

In [ ]:
# ============================================================
# 0. LIBRERÍAS Y CONFIGURACIÓN
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_white, het_breuschpagan
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de cambio de API de statsmodels
warnings.filterwarnings("ignore", message=".*rank-deficient.*")  # aparece en White por D^2 = D

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 140)

ALPHA = 0.05          # nivel de significancia para los tests
LAGS_BG = 4           # rezagos del Breusch-Godfrey (4 = un año en datos trimestrales)

# Columna de tiempo de la base apilada
COL_TIEMPO = "trimestre"   # formato "2010-T1": ordena bien como texto

### 0.1 Carga de la base
Se lee la base apilada que genera `construccion_variables.ipynb` y se **renombran** las columnas (por ejemplo, `u` pasa a llamarse `desempleo`) para que las salidas sean más fáciles de leer. El archivo CSV no se modifica.

Las columnas `ln_kaitz_c`, `kaitz_x_joven` y los rezagos se **eliminan y se vuelven a calcular** en la celda siguiente. En la base original los rezagos se calcularon antes de apilar los grupos, y aquí se calculan dentro de cada grupo.

La sentencia `assert` detiene el notebook con un mensaje claro si falta alguna variable.

In [ ]:
# ============================================================
# 0.1 CARGA DE DATOS
# ============================================================
# Base apilada generada por construccion_variables.ipynb (dos filas por trimestre)
df = pd.read_csv("datos_procesados/base_apilada.csv")

# Se renombran las columnas a los nombres usados en este notebook
df = df.rename(columns={
    "u": "desempleo", "o": "tasa_ocupacion", "ln_kaitz": "ln_Kaitz", "d_joven": "D_joven",
    "imacec_va": "IMACEC", "covid": "COVID", "q1": "Q1", "q2": "Q2", "q3": "Q3",
    "ln_sm_real": "ln_salario_real"})
# Se descartan las columnas ya calculadas en la base: se recalculan abajo dentro de cada grupo
df = df.drop(columns=["ln_kaitz_c", "kaitz_x_joven", "ln_kaitz_l1", "ln_kaitz_l2"], errors="ignore")

variables_requeridas = ["desempleo", "tasa_ocupacion", "ln_Kaitz", "D_joven", "IMACEC",
                        "inflacion", "COVID", "Q1", "Q2", "Q3", "ln_salario_real", "tendencia"]
faltantes = [v for v in variables_requeridas if v not in df.columns]
assert not faltantes, f"Faltan columnas en df: {faltantes}"
print(f"Observaciones totales: {len(df)}")
print(df["D_joven"].value_counts().rename({1: "Jóvenes (15-24)", 0: "Adultos (25+)"}))

### 0.2 Orden de los datos y variables derivadas
Hay tres pasos en esta celda, y los tres importan para que los resultados sean correctos:

1. **Orden por grupo y tiempo.** La base se ordena con los 67 trimestres de adultos primero y luego los 67 de jóvenes. Los tests de autocorrelación y los errores Newey-West comparan cada observación con la anterior, así que el orden tiene que ser cronológico dentro de cada grupo.
2. **Centrado del Kaitz.** Se calcula $\ln Kaitz^c = \ln Kaitz - \overline{\ln Kaitz}$. Sin centrar, la interacción $\ln Kaitz \times D_{joven}$ es casi igual a $D_{joven}$ multiplicado por una constante (≈ 8,3), lo que genera VIF muy altos. Centrar **no cambia β₁ ni β₃**; solo hace que β₂ sea la brecha de desempleo jóvenes–adultos con un Kaitz promedio.
3. **Rezagos dentro de cada grupo** (`groupby("D_joven").shift(k)`). Así el rezago del primer trimestre de jóvenes queda vacío (NaN) en lugar de tomar el último dato de adultos.

In [ ]:
# ============================================================
# 0.2 ORDEN TEMPORAL, ÍNDICE DE TIEMPO Y VARIABLES DERIVADAS
# ============================================================
df = df.copy()

# Índice de tiempo 't' (1, 2, 3, ...) dentro de cada grupo
if COL_TIEMPO is not None:
    df = df.sort_values(["D_joven", COL_TIEMPO])
df["t"] = df.groupby("D_joven").cumcount() + 1

# Orden: primero adultos, luego jóvenes; dentro de cada grupo, cronológico.
# Este orden es necesario para Breusch-Godfrey y para los errores HAC por grupo.
df = df.sort_values(["D_joven", "t"]).reset_index(drop=True)

# --- Centrado del Kaitz e interacción (reduce el VIF entre ln_Kaitz, D_joven e interacción) ---
MEDIA_KAITZ = df["ln_Kaitz"].mean()
df["ln_Kaitz_c"] = df["ln_Kaitz"] - MEDIA_KAITZ
df["interaccion_joven"] = df["ln_Kaitz_c"] * df["D_joven"]

# --- Lo mismo para el salario mínimo real (robustez 4.2) ---
df["ln_salario_real_c"] = df["ln_salario_real"] - df["ln_salario_real"].mean()
df["interaccion_joven_sr"] = df["ln_salario_real_c"] * df["D_joven"]

# --- Rezagos del Kaitz centrado: se calculan DENTRO de cada grupo para no mezclar series ---
for k in (1, 2):
    df[f"ln_Kaitz_c_L{k}"] = df.groupby("D_joven")["ln_Kaitz_c"].shift(k)
    df[f"interaccion_joven_L{k}"] = df[f"ln_Kaitz_c_L{k}"] * df["D_joven"]

print(f"Media de ln_Kaitz usada para centrar: {MEDIA_KAITZ:.4f}")
print(f"Correlación ln_Kaitz_c vs interacción: {df['ln_Kaitz_c'].corr(df['interaccion_joven']):.3f}")
df.head()

### Funciones auxiliares
Se definen una vez y se reutilizan en todo el notebook: extraer β₁, β₁+β₃ y β₃ con sus errores estándar y p-valores (bilaterales y unilaterales para H1: β>0 y H2: β₃>0), y traducirlos a puntos porcentuales.

**Qué hace cada función:**

| Función | Para qué sirve |
|---|---|
| `estimar()` | Estima el modelo OLS con errores clásicos o con errores robustos Newey-West. Con `hac-panel`, la corrección se calcula **por separado en cada grupo**, para no tratar el último trimestre de adultos como "vecino" del primero de jóvenes. |
| `efecto()` | Usa `t_test` para probar una **combinación de coeficientes**, por ejemplo β₁ + β₃ (efecto en jóvenes), que no aparece directamente en el `summary`. Devuelve el coeficiente, su error estándar, los p-valores y la traducción a puntos porcentuales. |
| `p_unilateral()` | H1 y H2 tienen dirección (β > 0), así que corresponde un test de **una cola**: si el t es positivo, p unilateral = p bilateral / 2. |
| `tabla_efectos()` | Junta en una tabla los tres efectos que pide la pauta: β₁, β₁+β₃ y β₃. |
| `maxlags_nw()` | Regla estándar de Newey-West para elegir cuántos rezagos considerar en los errores robustos. Con 67 trimestres da 3. |

In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================
CONTROLES = "IMACEC + inflacion + COVID + Q1 + Q2 + Q3"

def estimar(formula, data, cov="nonrobust", maxlags=None):
    '''Estima OLS con errores clásicos ('nonrobust'), Newey-West por grupo ('hac-panel')
    o Newey-West simple ('hac', para una sola serie).'''
    modelo = smf.ols(formula, data=data)
    if cov == "nonrobust":
        return modelo.fit()
    if cov == "hac-panel":
        # Newey-West calculado dentro de cada grupo (no mezcla el final de la serie de adultos
        # con el inicio de la serie de jóvenes). Requiere datos ordenados por grupo y tiempo.
        d = data.loc[modelo.data.row_labels]          # filas efectivamente usadas (sin NaN)
        return modelo.fit(cov_type="hac-panel",
                          cov_kwds={"groups": d["D_joven"].astype(int), "maxlags": maxlags})
    if cov == "hac":
        return modelo.fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    raise ValueError(cov)

def p_unilateral(t_stat, p_bilateral, signo=+1):
    '''p-valor unilateral para H: beta > 0 (signo=+1) o beta < 0 (signo=-1).'''
    return p_bilateral / 2 if np.sign(t_stat) == signo else 1 - p_bilateral / 2

def efecto(res, expr, nombre):
    '''Combina coeficientes (ej. 'ln_Kaitz_c + interaccion_joven') y devuelve una fila resumen.'''
    tt = res.t_test(f"{expr} = 0")
    b, se = float(np.squeeze(tt.effect)), float(np.squeeze(tt.sd))
    t, p = float(np.squeeze(tt.tvalue)), float(np.squeeze(tt.pvalue))
    return {"efecto": nombre, "beta": b, "ee": se, "t": t, "p_bilateral": p,
            "p_unilateral(beta>0)": p_unilateral(t, p, +1),
            "pp_ante_+1%": b / 100,                      # puntos porcentuales ante +1 %
            "pp_ante_+10%": b * np.log(1.10)}             # puntos porcentuales ante +10 %

def tabla_efectos(res, var="ln_Kaitz_c", inter="interaccion_joven"):
    '''Tabla con beta1 (adultos), beta1+beta3 (jóvenes) y beta3 (diferencia, H2).'''
    filas = [efecto(res, var, "β1: adultos"),
             efecto(res, f"{var} + {inter}", "β1+β3: jóvenes"),
             efecto(res, inter, "β3: diferencia jóvenes-adultos (H2)")]
    return pd.DataFrame(filas).set_index("efecto")

def maxlags_nw(T):
    '''Regla de Newey-West (1994) para el número de rezagos: floor(4*(T/100)^(2/9)).'''
    return int(np.floor(4 * (T / 100) ** (2 / 9)))

T_GRUPO = int(df.groupby("D_joven").size().min())
MAXLAGS = max(maxlags_nw(T_GRUPO), 1)
print(f"Observaciones por grupo: {T_GRUPO} -> rezagos Newey-West: {MAXLAGS}")

---
## 1. Modelo 1: principal con ambos grupos e interacción (4.1)

**Qué hace:** estima en una sola ecuación el efecto del salario mínimo sobre el desempleo de jóvenes y adultos, y prueba si el efecto es distinto entre ambos grupos. Se usa la base apilada: cada trimestre aparece dos veces, una vez por grupo. Sigue a Neumark y Wascher (2004) y a Brown et al. (1983).

$$u_{g,t} = \beta_0 + \beta_1 \ln Kaitz^c_t + \beta_2 D^{joven}_g + \beta_3 (\ln Kaitz^c_t \times D^{joven}_g) + \beta_4 IMACEC_t + \beta_5 \pi_t + \delta\, COVID_t + \textstyle\sum_s \gamma_s Q_s + \varepsilon_{g,t}$$

*g* = grupo (jóvenes o adultos), *t* = trimestre.

| Coeficiente | Variable | Interpretación | Signo esperado |
|---|---|---|---|
| β₀ | Intercepto | Desempleo adulto en oct–dic, sin COVID, con Kaitz promedio e Imacec e inflación en 0 | + |
| **β₁** | `ln_Kaitz_c` | **Efecto del Kaitz en adultos.** Un alza de 10 % del Kaitz cambia el desempleo adulto en β₁/10 pp | + (H1) |
| β₂ | `D_joven` | Cuántos pp más alto es el desempleo juvenil que el adulto, con un Kaitz promedio | + |
| **β₃** | `interaccion_joven` | **Cuánto más afecta el Kaitz a los jóvenes que a los adultos.** Prueba H2 | + (H2) |
| **β₁ + β₃** | (combinación) | **Efecto del Kaitz en jóvenes.** No aparece en el `summary`; se calcula con `t_test` | + (H1) |
| β₄ | `IMACEC` | pp de desempleo por cada punto adicional de crecimiento económico | − |
| β₅ | `inflacion` | pp de desempleo por cada punto adicional de inflación | ambiguo |
| δ | `COVID` | pp adicionales de desempleo durante la pandemia | + |
| γ₁, γ₂, γ₃ | `Q1`, `Q2`, `Q3` | Diferencia estacional en pp respecto de oct–dic | ambiguo |

**¿Por qué el Kaitz está centrado?** Al restarle su media, β₂ se lee con un Kaitz **promedio** (sin centrar, se leería con ln Kaitz = 0, que no tiene sentido económico) y la interacción deja de estar casi perfectamente correlacionada con `D_joven`. β₁ y β₃ **no cambian** con el centrado.

**Cómo se prueban las hipótesis:**
- **H1** (el salario mínimo aumenta el desempleo): β₁ > 0 en adultos y β₁ + β₃ > 0 en jóvenes.
- **H2** (el efecto es mayor en jóvenes): β₃ > 0.

Ambas se prueban con p-valores de **una cola**, porque las hipótesis tienen dirección.

In [ ]:
# ============================================================
# 1. MODELO PRINCIPAL (OLS con errores estándar clásicos)
# ============================================================
F_PRINCIPAL = f"desempleo ~ ln_Kaitz_c + D_joven + interaccion_joven + {CONTROLES}"

res_principal = estimar(F_PRINCIPAL, df)
print(res_principal.summary())

**Cómo leer el `summary`:**
- `coef` es el coeficiente estimado; `P>|t|` es el p-valor bilateral (si es menor que 0,05, el coeficiente es significativo).
- `ln_Kaitz_c` = **β₁**, el efecto en adultos. `interaccion_joven` = **β₃**, cuánto más afecta a los jóvenes. `D_joven` = **β₂**, la brecha de desempleo entre jóvenes y adultos con un Kaitz promedio.
- Al final del `summary`, el **Durbin-Watson** da una primera pista de autocorrelación: un valor lejos de 2 indica problemas.
- **Ojo:** estos errores estándar todavía son los clásicos. Los definitivos son los de la sección 3.6.

La celda siguiente calcula también **β₁ + β₃**, el efecto en jóvenes, que el `summary` no muestra.

In [ ]:
# Efectos clave para la interpretación (errores clásicos; ver sección 3.6 para la versión HAC)
print("Efecto del Kaitz sobre el desempleo (pp = puntos porcentuales)\n")
display(tabla_efectos(res_principal))

---
## 2. Modelos 2 y 3: regresiones separadas por grupo (4.2)

**Qué hacen:** estiman el mismo modelo **por separado** para jóvenes (modelo 2) y para adultos (modelo 3), sin la dummy de edad ni la interacción. Son el complemento que pide la pauta, siguiendo a Brown et al. (1983), y muestran el efecto en cada grupo de forma más directa.

$$u_t = \alpha_0 + \alpha_1 \ln Kaitz^c_t + \alpha_2 IMACEC_t + \alpha_3 \pi_t + \delta\, COVID_t + \textstyle\sum_s \gamma_s Q_s + \varepsilon_t$$

| Coeficiente | Variable | Interpretación |
|---|---|---|
| α₀ | Intercepto | Desempleo del grupo en oct–dic, sin COVID, con Kaitz promedio |
| **α₁** | `ln_Kaitz_c` | **Efecto del Kaitz en ese grupo:** un alza de 10 % cambia su desempleo en α₁/10 pp |
| α₂, α₃, δ, γ | Controles | Igual que en el modelo 1, pero ahora **cada grupo tiene sus propios valores** |

**Diferencia con el modelo 1:** en el modelo 1, los controles (Imacec, inflación, COVID, trimestres) tienen el **mismo efecto** en ambos grupos; aquí cada grupo tiene el suyo. Por eso α₁ de jóvenes no tiene por qué coincidir exactamente con β₁ + β₃, ni α₁ de adultos con β₁. Si son parecidos, el modelo 1 es una buena simplificación.

**Limitación:** comparar los dos α₁ es intuitivo, pero no es una prueba formal de la diferencia. La prueba formal de H2 es β₃ del modelo 1.

In [ ]:
# ============================================================
# 2. REGRESIONES SEPARADAS POR GRUPO
# ============================================================
F_GRUPO = f"desempleo ~ ln_Kaitz_c + {CONTROLES}"

df_jov = df[df["D_joven"] == 1].copy()
df_adu = df[df["D_joven"] == 0].copy()

res_jov = estimar(F_GRUPO, df_jov)
res_adu = estimar(F_GRUPO, df_adu)

print("=" * 30, "JÓVENES (15-24)", "=" * 30)
print(res_jov.summary())
print("\n" + "=" * 30, "ADULTOS (25+)", "=" * 30)
print(res_adu.summary())

Esta tabla compara el coeficiente del Kaitz en las dos regresiones separadas. Si el de jóvenes es mayor, apoya H2 de forma intuitiva. Sin embargo, la **prueba formal** de H2 es β₃ en el modelo con interacción, porque ahí se testea directamente la diferencia entre los grupos.

In [ ]:
# Comparación directa del efecto del Kaitz en cada grupo
comp_grupos = pd.DataFrame([
    {**efecto(res_jov, "ln_Kaitz_c", "Jóvenes (15-24)"), "R2": res_jov.rsquared, "N": int(res_jov.nobs)},
    {**efecto(res_adu, "ln_Kaitz_c", "Adultos (25+)"),   "R2": res_adu.rsquared, "N": int(res_adu.nobs)},
]).set_index("efecto")
display(comp_grupos)

---
## 3. Supuestos y tests econométricos (4.3)
Todos los tests se aplican al **modelo principal**. Regla de lectura: si p < 0,05 se rechaza la hipótesis nula.

### 3.1 Normalidad: Jarque-Bera
Compara la asimetría y la curtosis de los residuos con las de una distribución normal (asimetría 0, curtosis 3).
- **H0:** los residuos son normales.
- Si se rechaza, no es grave: con más de 100 observaciones, el teorema central del límite hace que los tests t sigan siendo aproximadamente válidos. Basta mencionarlo en el informe.

In [ ]:
# ============================================================
# 3.1 JARQUE-BERA (normalidad de los residuos)
# H0: los residuos son normales
# ============================================================
jb_stat, jb_p, asimetria, curtosis = jarque_bera(res_principal.resid)
print(f"Jarque-Bera = {jb_stat:.3f} | p-valor = {jb_p:.4f}")
print(f"Asimetría = {asimetria:.3f} | Curtosis = {curtosis:.3f} (normal = 3)")
print("-> Se rechaza normalidad." if jb_p < ALPHA else "-> No se rechaza normalidad.")
print("   (Con ~130 observaciones la inferencia es aproximadamente válida aunque no haya normalidad.)")

### 3.2 Multicolinealidad: VIF
El VIF de una variable mide cuánto se infla la varianza de su coeficiente porque esa variable se parece a las demás. Se calcula como $VIF_j = 1/(1-R^2_j)$, donde $R^2_j$ sale de regresionar la variable *j* contra el resto.
- Un VIF mayor que 10 indica un problema serio, y uno mayor que 5 merece atención.
- La segunda parte de la celda calcula el VIF **sin centrar** el Kaitz. Sirve para mostrar en el informe por qué se centró: los VIF de `D_joven` y de la interacción bajan muchísimo.

In [ ]:
# ============================================================
# 3.2 VIF (multicolinealidad)
# Regla práctica: VIF > 10 indica multicolinealidad preocupante (> 5 merece atención)
# ============================================================
X = pd.DataFrame(res_principal.model.exog, columns=res_principal.model.exog_names)
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
}).query("variable != 'Intercept'").set_index("variable")
display(vif.sort_values("VIF", ascending=False))

# Comparación: VIF de la interacción SIN centrar, para justificar el centrado en el informe
X_sin = X.copy()
X_sin["ln_Kaitz_c"] = df["ln_Kaitz"].values
X_sin["interaccion_joven"] = (df["ln_Kaitz"] * df["D_joven"]).values
vif_sin = {c: variance_inflation_factor(X_sin.values, list(X_sin.columns).index(c))
           for c in ["ln_Kaitz_c", "D_joven", "interaccion_joven"]}
print("VIF sin centrar el Kaitz (solo como comparación):")
print({k.replace("_c", ""): round(v, 1) for k, v in vif_sin.items()})

### 3.3 Autocorrelación: Breusch-Godfrey
Regresiona los residuos contra las variables del modelo y contra sus propios rezagos (hasta 4). Si los rezagos explican los residuos, el error de un trimestre "arrastra" información al siguiente.
- **H0:** no hay autocorrelación.
- Es muy común en series de tiempo: el desempleo cambia lentamente, así que si un trimestre está por sobre lo que predice el modelo, el siguiente probablemente también.
- **Consecuencia:** los coeficientes siguen siendo insesgados, pero los errores estándar clásicos quedan **subestimados**, lo que hace que todo parezca más significativo de lo que es. Por eso, si se rechaza H0, se usan errores Newey-West (sección 3.6).
- Se reporta el test para el modelo completo y además para cada grupo por separado, para confirmar que el resultado no se debe a la unión entre las dos series.

In [ ]:
# ============================================================
# 3.3 BREUSCH-GODFREY (autocorrelación)
# H0: no hay autocorrelación hasta el rezago LAGS_BG
# ============================================================
# (a) Sobre el modelo principal apilado (df ordenado por grupo y tiempo)
bg_lm, bg_lm_p, bg_f, bg_f_p = acorr_breusch_godfrey(res_principal, nlags=LAGS_BG)
print(f"Modelo principal: LM = {bg_lm:.3f} (p = {bg_lm_p:.4f}) | F = {bg_f:.3f} (p = {bg_f_p:.4f})")

# (b) Chequeo por grupo: evita que el 'empalme' entre la serie de adultos y la de jóvenes
#     contamine el test. Útil como respaldo en el informe.
for nombre, r in [("Jóvenes", res_jov), ("Adultos", res_adu)]:
    lm, lm_p, _, _ = acorr_breusch_godfrey(r, nlags=LAGS_BG)
    print(f"{nombre:8s}: LM = {lm:.3f} (p = {lm_p:.4f})")

HAY_AUTOCORRELACION = bg_lm_p < ALPHA
print("\n-> Hay autocorrelación." if HAY_AUTOCORRELACION else "\n-> No se rechaza ausencia de autocorrelación.")

### 3.4 Heterocedasticidad: White
Revisa si la varianza del error cambia según los valores de las variables explicativas. Lo hace regresionando los residuos al cuadrado contra las variables, sus cuadrados y sus productos.
- **H0:** homocedasticidad (varianza constante).
- En esta base es esperable que la varianza sea distinta entre jóvenes y adultos, porque el desempleo juvenil es más volátil.
- Breusch-Pagan se incluye como respaldo, porque es más estable cuando el modelo tiene muchas dummies.
- **Consecuencia:** la misma que en la autocorrelación, y la solución también: errores Newey-West, que corrigen ambos problemas a la vez.

In [ ]:
# ============================================================
# 3.4 WHITE (heterocedasticidad)
# H0: homocedasticidad
# ============================================================
exog = res_principal.model.exog
try:
    w_lm, w_lm_p, w_f, w_f_p = het_white(res_principal.resid, exog)
    print(f"White: LM = {w_lm:.3f} (p = {w_lm_p:.4f}) | F = {w_f:.3f} (p = {w_f_p:.4f})")
    p_het = w_lm_p
except Exception as e:
    # Con muchas dummies el test de White puede quedar con regresores redundantes (D^2 = D).
    print(f"White no pudo calcularse ({e}). Se usa Breusch-Pagan como alternativa.")
    p_het = None

# Nota: con dummies, het_white incluye términos redundantes (D^2 = D), lo que infla un poco
# los grados de libertad y hace el test algo conservador.
# Breusch-Pagan como complemento (más estable con dummies)
bp_lm, bp_lm_p, _, _ = het_breuschpagan(res_principal.resid, exog)
print(f"Breusch-Pagan: LM = {bp_lm:.3f} (p = {bp_lm_p:.4f})")

HAY_HETEROCEDASTICIDAD = (p_het if p_het is not None else bp_lm_p) < ALPHA
print("-> Hay heterocedasticidad." if HAY_HETEROCEDASTICIDAD else "-> No se rechaza homocedasticidad.")

### 3.5 Raíz unitaria: Augmented Dickey-Fuller (va en el anexo)
Prueba si una serie es **estacionaria**, es decir, si vuelve a una media estable, o si tiene **raíz unitaria** y deriva sin volver.
- **H0:** la serie tiene raíz unitaria (no es estacionaria). Si p < 0,05, la serie es estacionaria.
- Se prueban dos especificaciones: solo con constante, y con constante más tendencia. Una serie puede ser estacionaria alrededor de una tendencia.
- El test se hace **por grupo**, porque la base apilada no es una serie de tiempo única.
- Al final se repite el test en **primeras diferencias** para las series no estacionarias. Si la diferencia sí es estacionaria, la serie es I(1).

**Por qué importa:** si el desempleo y el Kaitz tienen raíz unitaria, podrían aparecer relacionados solo porque ambos tienen tendencia (**regresión espuria**). Por eso en la sección 4.5 se agrega una robustez con tendencia lineal.

In [ ]:
# ============================================================
# 3.5 ADF (raíz unitaria) — para el ANEXO
# H0: la serie tiene raíz unitaria (no estacionaria)
# Cada serie se testea POR GRUPO (las series apiladas no son una serie de tiempo).
# Las variables comunes a ambos grupos (Kaitz, IMACEC, inflación) se testean una sola vez.
# ============================================================
def adf(serie, regresion="c"):
    est, p, rez, nobs, crit, _ = adfuller(serie.dropna(), regression=regresion, autolag="AIC")
    return {"ADF": est, "p_valor": p, "rezagos": rez, "N": nobs, "crit_5%": crit["5%"],
            "conclusión": "Estacionaria" if p < ALPHA else "Raíz unitaria"}

filas_adf = []
for grupo, d in [("Jóvenes", df_jov), ("Adultos", df_adu)]:
    for v in ["desempleo", "tasa_ocupacion"]:
        for reg, lab in [("c", "constante"), ("ct", "const.+tendencia")]:
            filas_adf.append({"serie": f"{v} ({grupo})", "especificación": lab, **adf(d[v], reg)})

for v in ["ln_Kaitz", "ln_salario_real", "IMACEC", "inflacion"]:
    for reg, lab in [("c", "constante"), ("ct", "const.+tendencia")]:
        filas_adf.append({"serie": v, "especificación": lab, **adf(df_adu[v], reg)})

tabla_adf = pd.DataFrame(filas_adf).set_index(["serie", "especificación"])
display(tabla_adf)

# Opcional: ADF en primeras diferencias para las series con raíz unitaria
print("ADF en primeras diferencias (solo series no estacionarias en niveles, especificación con constante):")
for (serie, esp), fila in tabla_adf.iterrows():
    if esp == "constante" and fila["conclusión"] == "Raíz unitaria":
        nombre, _, grupo = serie.partition(" (")
        base = df_jov if "Jóvenes" in grupo else df_adu
        r = adf(base[nombre].diff())
        print(f"  Δ{serie:30s} ADF = {r['ADF']:7.3f}  p = {r['p_valor']:.4f}  -> {r['conclusión']}")

### 3.6 Reestimación con errores estándar robustos HAC (Newey-West)
Si Breusch-Godfrey **o** White rechazan, el modelo principal se reestima con errores Newey-West. Como la base apila dos series, se usa `cov_type="hac-panel"`: la corrección Newey-West se calcula **dentro de cada grupo**, así que no trata el último trimestre de adultos como "vecino" del primero de jóvenes. Los coeficientes no cambian; solo cambian los errores estándar, los t y los p-valores.

**Ojo al leer el `summary`:** con `hac-panel` y solo 2 grupos, el *F global* y su p-valor no son confiables; para el informe usa los z/p de cada coeficiente y los efectos de la tabla.

La variable `COV_ELEGIDO` queda definida aquí y se usa en toda la sección de robustez.

**Qué modelos se reestiman aquí:** el modelo 1 (principal) y los modelos 2 y 3 (por grupo). Los coeficientes son **los mismos** de las secciones 1 y 2; solo cambian los errores estándar, los estadísticos z y los p-valores. **Estos son los resultados que se reportan en el informe.**

- Modelo 1: Newey-West calculado **dentro de cada grupo** (`hac-panel`).
- Modelos 2 y 3: Newey-West estándar (`HAC`), porque cada uno es una sola serie de tiempo.

En el `summary` con errores robustos, la columna `t` pasa a llamarse `z` y `P>|t|` pasa a `P>|z|`. Se leen igual.

In [ ]:
# ============================================================
# 3.6 BLOQUE CONDICIONAL: ERRORES HAC (NEWEY-WEST)
# ============================================================
if HAY_AUTOCORRELACION or HAY_HETEROCEDASTICIDAD:
    COV_ELEGIDO = "hac-panel"
    motivo = " y ".join(m for m, c in [("autocorrelación", HAY_AUTOCORRELACION),
                                        ("heterocedasticidad", HAY_HETEROCEDASTICIDAD)] if c)
    print(f"Se detectó {motivo} -> reestimación con errores Newey-West (maxlags = {MAXLAGS}).\n")
else:
    COV_ELEGIDO = "nonrobust"
    print("No se detectaron problemas -> se mantienen los errores clásicos.\n")

res_final = estimar(F_PRINCIPAL, df, cov=COV_ELEGIDO, maxlags=MAXLAGS)
print(res_final.summary())

# Errores HAC también para las regresiones por grupo (Newey-West simple: cada una es una serie)
COV_GRUPO = "hac" if COV_ELEGIDO != "nonrobust" else "nonrobust"
res_jov_f = estimar(F_GRUPO, df_jov, cov=COV_GRUPO, maxlags=MAXLAGS)
res_adu_f = estimar(F_GRUPO, df_adu, cov=COV_GRUPO, maxlags=MAXLAGS)

Esta tabla pone lado a lado los errores estándar clásicos y los robustos. **Los coeficientes son idénticos**; solo cambian los errores estándar y los p-valores. Si un coeficiente sigue siendo significativo con errores robustos, el resultado es confiable a pesar de la autocorrelación y la heterocedasticidad. Esta comparación es útil para el texto de la sección 4.3 del informe.

In [ ]:
# Comparación de errores estándar: clásicos vs. elegidos
comparacion_ee = pd.DataFrame({
    "coef": res_principal.params,
    "ee_clásico": res_principal.bse,
    "p_clásico": res_principal.pvalues,
    f"ee_{COV_ELEGIDO}": res_final.bse,
    f"p_{COV_ELEGIDO}": res_final.pvalues,
})
display(comparacion_ee)

print("\nEFECTOS CLAVE (modelo principal, errores finales):")
display(tabla_efectos(res_final))

---
## 4. Robustez (4.4): modelos 4 a 9

**Idea general:** un resultado es **robusto** si se mantiene (mismo signo y significancia) cuando se cambian decisiones razonables del análisis. Cada modelo de esta sección cambia **una sola cosa** respecto del modelo 1 y deja todo lo demás igual. Así, si el resultado cambia, se sabe qué lo provocó.

| Modelo | Qué cambia respecto del modelo 1 | Qué se espera si el resultado es robusto |
|---|---|---|
| 4 | La dependiente pasa a ser la tasa de ocupación | Coeficientes del Kaitz **negativos** |
| 5 | El Kaitz se reemplaza por el salario mínimo real | β₁ y β₃ positivos |
| 6 y 7 | El Kaitz se usa con 1 y 2 trimestres de rezago | β₁ y β₃ positivos, de magnitud parecida |
| 8 | Se eliminan los trimestres COVID | β₁ y β₃ parecidos al modelo 1 |
| 9 | Se agrega una tendencia lineal | β₁ y β₃ siguen siendo positivos y significativos |

Todos usan los mismos errores estándar elegidos en 3.6 (`COV_ELEGIDO`). La interpretación de β₁, β₃ y β₁ + β₃ es la misma del modelo 1, salvo en el modelo 4, donde la dependiente es la ocupación.

> La pauta también pide **separar 15–19 vs. 20–24 años**. El archivo actual del INE solo trae el tramo 15–24: hay que descargar esos dos subtramos de la ENE, agregarlos en `construccion_variables.ipynb` y repetir la sección 1 con una dummy `D_15_19` sobre la submuestra de jóvenes.

### 4.1 Modelo 4: tasa de ocupación como variable dependiente

**Por qué:** la literatura prefiere esta medida porque un alza del mínimo puede hacer que algunos jóvenes **dejen de buscar trabajo**. Esas personas salen de la fuerza laboral y entonces no cuentan como desempleados, así que la tasa de desempleo **subestima** el efecto (Brown et al., 1983).

$$o_{g,t} = \beta_0 + \beta_1 \ln Kaitz^c_t + \beta_2 D^{joven}_g + \beta_3 (\ln Kaitz^c_t \times D^{joven}_g) + \text{controles} + \varepsilon_{g,t}$$

| Coeficiente | Interpretación | Signo esperado |
|---|---|---|
| β₁ | Un alza de 10 % del Kaitz cambia la **ocupación adulta** en β₁/10 pp | − |
| β₁ + β₃ | Un alza de 10 % del Kaitz cambia la **ocupación juvenil** en (β₁+β₃)/10 pp | − |
| β₃ | Cuánto **más cae** la ocupación juvenil que la adulta. Aquí H2 se lee al revés: se espera β₃ < 0 | − |
| β₂ | Diferencia de ocupación entre jóvenes y adultos con un Kaitz promedio | − |

**Elasticidad.** La celda calcula además $\varepsilon = \beta / \bar{o}$: el cambio **porcentual** de la ocupación ante un alza de 1 % del Kaitz. Es la medida comparable con el rango de −0,1 a −0,3 de Brown et al. (1982). Una elasticidad mucho mayor en valor absoluto indica que el modelo podría estar capturando otra cosa, como una tendencia.

**Cómo usarlo en el informe:** si la caída de la ocupación juvenil (en pp) es mayor que el alza del desempleo juvenil, la diferencia corresponde a jóvenes que **salieron de la fuerza laboral**.

In [ ]:
# ============================================================
# 4.1 ROBUSTEZ: TASA DE OCUPACIÓN COMO DEPENDIENTE
# Signo esperado: NEGATIVO (más salario mínimo -> menos ocupación).
# ============================================================
F_OCUP = f"tasa_ocupacion ~ ln_Kaitz_c + D_joven + interaccion_joven + {CONTROLES}"
res_ocup = estimar(F_OCUP, df, cov=COV_ELEGIDO, maxlags=MAXLAGS)
print(res_ocup.summary())

tab_ocup = tabla_efectos(res_ocup)
# Elasticidad empleo-salario mínimo (comparable con -0,1 a -0,3 de Brown et al., 1982):
# elasticidad = (d ocupación / ocupación) / (d ln Kaitz) = beta / media de la tasa de ocupación del grupo
media_ocup_adu = df_adu["tasa_ocupacion"].mean()
media_ocup_jov = df_jov["tasa_ocupacion"].mean()
tab_ocup["elasticidad"] = [tab_ocup.loc["β1: adultos", "beta"] / media_ocup_adu,
                           tab_ocup.loc["β1+β3: jóvenes", "beta"] / media_ocup_jov,
                           np.nan]
tab_ocup["p_unilateral(beta<0)"] = 1 - tab_ocup["p_unilateral(beta>0)"]
print("\nEfecto del Kaitz sobre la tasa de ocupación:")
display(tab_ocup)

### 4.2 Modelo 5: salario mínimo real en lugar del Kaitz

**Por qué:** el Kaitz mide el salario mínimo **en relación con el salario promedio**; el salario real lo mide en **poder de compra** (pesos de 2023). Si el resultado se mantiene con esta medida, no depende de cómo se defina el salario mínimo.

$$u_{g,t} = \beta_0 + \beta_1 \ln SMreal^c_t + \beta_2 D^{joven}_g + \beta_3 (\ln SMreal^c_t \times D^{joven}_g) + \text{controles} + \varepsilon_{g,t}$$

| Coeficiente | Variable | Interpretación |
|---|---|---|
| β₁ | `ln_salario_real_c` | Un alza de 10 % del salario mínimo real cambia el desempleo **adulto** en β₁/10 pp |
| β₃ | `interaccion_joven_sr` | Cuánto más afecta a los jóvenes (H2) |
| β₁ + β₃ | (combinación) | Efecto en el desempleo **juvenil** |

El salario real también se centra en su media y se interactúa con `D_joven`, igual que el Kaitz. Las magnitudes no se comparan directamente con las del Kaitz, porque un alza de 10 % en el salario real no equivale a un alza de 10 % en el Kaitz.

In [ ]:
# ============================================================
# 4.2 ROBUSTEZ: SALARIO MÍNIMO REAL EN VEZ DEL KAITZ
# (también centrado, con su propia interacción)
# ============================================================
F_SR = f"desempleo ~ ln_salario_real_c + D_joven + interaccion_joven_sr + {CONTROLES}"
res_sr = estimar(F_SR, df, cov=COV_ELEGIDO, maxlags=MAXLAGS)
print(res_sr.summary())
display(tabla_efectos(res_sr, var="ln_salario_real_c", inter="interaccion_joven_sr"))

### 4.3 Modelos 6 y 7: Kaitz rezagado 1 y 2 trimestres

**Por qué:** las empresas pueden demorar en ajustar sus contrataciones después de un reajuste. Además, usar el Kaitz de trimestres anteriores reduce el riesgo de **causalidad inversa**: el desempleo de hoy no puede haber influido en el salario mínimo de hace 3 o 6 meses.

$$u_{g,t} = \beta_0 + \beta_1 \ln Kaitz^c_{t-k} + \beta_2 D^{joven}_g + \beta_3 (\ln Kaitz^c_{t-k} \times D^{joven}_g) + \text{controles} + \varepsilon_{g,t}, \quad k = 1, 2$$

| Coeficiente | Variable | Interpretación |
|---|---|---|
| β₁ | `ln_Kaitz_c_L1` / `ln_Kaitz_c_L2` | Un alza de 10 % del Kaitz hace *k* trimestres cambia el desempleo **adulto** de hoy en β₁/10 pp |
| β₃ | `interaccion_joven_L1` / `_L2` | Cuánto más afecta a los jóvenes (H2) |
| β₁ + β₃ | (combinación) | Efecto rezagado en el desempleo **juvenil** |

Se pierden 1 o 2 observaciones por grupo, porque al inicio de la muestra no existe el trimestre anterior. **Lectura:** si los coeficientes rezagados son parecidos a los del modelo 1, el efecto es persistente. Si son mayores, el efecto tarda en aparecer.

In [ ]:
# ============================================================
# 4.3 ROBUSTEZ: KAITZ REZAGADO 1 Y 2 TRIMESTRES
# Los rezagos se construyeron por grupo (sección 0.2). Se pierden 1 o 2 obs. por grupo.
# ============================================================
res_lag = {}
for k in (1, 2):
    F_LAG = f"desempleo ~ ln_Kaitz_c_L{k} + D_joven + interaccion_joven_L{k} + {CONTROLES}"
    d_lag = df.dropna(subset=[f"ln_Kaitz_c_L{k}"])
    res_lag[k] = estimar(F_LAG, d_lag, cov=COV_ELEGIDO, maxlags=MAXLAGS)
    print("=" * 25, f"Kaitz rezagado {k} trimestre(s) | N = {int(res_lag[k].nobs)}", "=" * 25)
    print(res_lag[k].summary().tables[1])
    display(tabla_efectos(res_lag[k], var=f"ln_Kaitz_c_L{k}", inter=f"interaccion_joven_L{k}"))

### 4.4 Modelo 8: excluyendo el periodo COVID

**Por qué:** la pandemia provocó cambios extremos en el desempleo que no tienen relación con el salario mínimo. El modelo 1 ya incluye la dummy COVID, pero aquí se **eliminan** esos trimestres (2020-T2 a 2021-T2, 5 por grupo) para verificar que no son ellos los que determinan el resultado.

$$u_{g,t} = \beta_0 + \beta_1 \ln Kaitz^c_t + \beta_2 D^{joven}_g + \beta_3 (\ln Kaitz^c_t \times D^{joven}_g) + \beta_4 IMACEC_t + \beta_5 \pi_t + \textstyle\sum_s \gamma_s Q_s + \varepsilon_{g,t}, \quad COVID_t = 0$$

Las variables y su interpretación son **las mismas del modelo 1**, excepto que la dummy COVID se quita de la fórmula: en esta submuestra vale siempre 0, así que no se puede estimar. **Lectura:** si β₁ y β₃ son parecidos a los del modelo 1, la pandemia no está determinando el resultado.

In [ ]:
# ============================================================
# 4.4 ROBUSTEZ: EXCLUYENDO EL PERIODO COVID
# La dummy COVID se elimina de la fórmula porque vale 0 en toda la submuestra.
# ============================================================
df_nocovid = df[df["COVID"] == 0].copy()
CONTROLES_SIN_COVID = CONTROLES.replace("COVID + ", "")
F_NOCOVID = f"desempleo ~ ln_Kaitz_c + D_joven + interaccion_joven + {CONTROLES_SIN_COVID}"

# Nota: el Kaitz se mantiene centrado con la media de la muestra completa (β1 y β3 no cambian
# por esto; solo cambia la lectura de β2).
res_nocovid = estimar(F_NOCOVID, df_nocovid, cov=COV_ELEGIDO, maxlags=MAXLAGS)
print(f"Observaciones sin COVID: {int(res_nocovid.nobs)}")
print(res_nocovid.summary())
display(tabla_efectos(res_nocovid))

### 4.5 Modelo 9: con tendencia lineal (robustez adicional)

**Por qué:** el test ADF (sección 3.5) mostró que el desempleo y el Kaitz no son estacionarios en niveles. Si ambos crecen en el tiempo por razones distintas, el modelo podría encontrar una relación que en realidad es solo una **tendencia común** (regresión espuria).

$$u_{g,t} = \beta_0 + \beta_1 \ln Kaitz^c_t + \beta_2 D^{joven}_g + \beta_3 (\ln Kaitz^c_t \times D^{joven}_g) + \text{controles} + \theta \cdot tendencia_t + \varepsilon_{g,t}$$

| Coeficiente | Variable | Interpretación |
|---|---|---|
| β₁, β₃, β₁ + β₃ | Igual que en el modelo 1 | Efecto del Kaitz **una vez descontada la tendencia lineal** |
| θ | `tendencia` | pp que cambia el desempleo por cada trimestre que pasa, por factores no incluidos en el modelo |

**Lectura:** si β₁ y β₃ siguen siendo positivos y significativos, el Kaitz explica el desempleo **más allá** de la tendencia común. Ojo: el Kaitz está muy correlacionado con el tiempo, así que los errores estándar de este modelo son más grandes y los coeficientes pueden cambiar bastante de magnitud.

In [ ]:
# ============================================================
# 4.5 ROBUSTEZ ADICIONAL: TENDENCIA LINEAL
# El ADF indica que el desempleo y el Kaitz no son estacionarios en niveles: ambos podrían
# moverse juntos solo por tener tendencia (regresión espuria). Se agrega una tendencia lineal
# para verificar que el efecto del Kaitz no la está capturando.
# ============================================================
F_TEND = f"desempleo ~ ln_Kaitz_c + D_joven + interaccion_joven + {CONTROLES} + tendencia"
res_tend = estimar(F_TEND, df, cov=COV_ELEGIDO, maxlags=MAXLAGS)
print(res_tend.summary())
display(tabla_efectos(res_tend))

---
## 5. Resultados para el informe

### 5.0 Tabla de coeficientes del modelo principal, con interpretación
Esta tabla resume **cada variable de la regresión principal**: qué representa, su coeficiente, su error estándar (ya corregido con Newey-West si correspondía), su p-valor y cómo se interpreta. Al final se agrega el **efecto en jóvenes (β₁+β₃)**, que no aparece como coeficiente en el `summary` porque es una combinación de dos coeficientes.

**Cómo se interpreta cada tipo de variable** (la dependiente es la tasa de desempleo, en %):
- **Kaitz (en logaritmo):** modelo nivel-log, así que β × ln(1,10) ≈ β/10 son los puntos porcentuales (pp) que cambia el desempleo si el Kaitz sube 10 %.
- **Imacec e inflación (en %):** β son los pp que cambia el desempleo si la variable sube 1 punto.
- **Dummies (joven, COVID, trimestres):** β son los pp de diferencia respecto del grupo base (adultos, periodo sin COVID, trimestre oct–dic).

Las interpretaciones se escriben a partir de los valores estimados, así que se actualizan solas si cambian los datos. Si un coeficiente no es significativo al 10 %, la tabla lo indica.

In [ ]:
# ============================================================
# 5.0 TABLA DE COEFICIENTES CON INTERPRETACIÓN (modelo principal, errores finales)
# ============================================================
LN10 = np.log(1.10)   # alza de 10 %

def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

def nota_sig(p):
    if p < 0.05:
        return ""
    if p < 0.10:
        return " (significativo solo al 10 %: evidencia débil)"
    return " (no significativo: no se puede afirmar que el efecto sea distinto de 0)"

def signo(b, sube="aumenta", baja="disminuye"):
    return sube if b > 0 else baja

# Descripción de cada variable y función que redacta su interpretación a partir de (b, p)
variables = {
    "Intercept": ("Constante (β₀)",
        lambda b, p: f"Desempleo esperado de los adultos ({b:.2f} %) en oct–dic, sin COVID, "
                     f"con Kaitz en su promedio e Imacec e inflación en 0."),
    "ln_Kaitz_c": ("Log del índice de Kaitz, centrado (β₁)",
        lambda b, p: f"ADULTOS: si el Kaitz sube 10 %, el desempleo {signo(b)} "
                     f"{abs(b*LN10):.2f} pp{nota_sig(p)}."),
    "D_joven": ("Dummy joven, 1 = 15–24 años (β₂)",
        lambda b, p: f"Con un Kaitz promedio, el desempleo de los jóvenes es {abs(b):.2f} pp "
                     f"{'mayor' if b > 0 else 'menor'} que el de los adultos{nota_sig(p)}."),
    "interaccion_joven": ("Kaitz centrado × joven (β₃, prueba H2)",
        lambda b, p: f"Ante un alza de 10 % del Kaitz, el desempleo juvenil sube {abs(b*LN10):.2f} pp "
                     f"{'más' if b > 0 else 'menos'} que el adulto{nota_sig(p)}."),
    "IMACEC": ("Variación anual del Imacec, % (β₄)",
        lambda b, p: f"Si la economía crece 1 punto más, el desempleo {signo(b)} "
                     f"{abs(b):.2f} pp{nota_sig(p)}."),
    "inflacion": ("Inflación anual, % (β₅)",
        lambda b, p: f"Si la inflación sube 1 punto, el desempleo {signo(b)} "
                     f"{abs(b):.2f} pp{nota_sig(p)}."),
    "COVID": ("Dummy COVID, 2020-T2 a 2021-T2 (δ)",
        lambda b, p: f"Durante la pandemia el desempleo fue {abs(b):.2f} pp "
                     f"{'mayor' if b > 0 else 'menor'} de lo que predicen las demás variables{nota_sig(p)}."),
    "Q1": ("Dummy trimestre ene–mar (γ₁)",
        lambda b, p: f"En ene–mar el desempleo es {abs(b):.2f} pp {'mayor' if b > 0 else 'menor'} "
                     f"que en oct–dic{nota_sig(p)}."),
    "Q2": ("Dummy trimestre abr–jun (γ₂)",
        lambda b, p: f"En abr–jun el desempleo es {abs(b):.2f} pp {'mayor' if b > 0 else 'menor'} "
                     f"que en oct–dic{nota_sig(p)}."),
    "Q3": ("Dummy trimestre jul–sep (γ₃)",
        lambda b, p: f"En jul–sep el desempleo es {abs(b):.2f} pp {'mayor' if b > 0 else 'menor'} "
                     f"que en oct–dic{nota_sig(p)}."),
}

filas = []
for var, (descripcion, interpretar) in variables.items():
    b, ee, p = res_final.params[var], res_final.bse[var], res_final.pvalues[var]
    filas.append({"variable": var, "descripción": descripcion,
                  "coef": f"{b:.3f}{estrellas(p)}", "ee": round(ee, 3), "p_valor": round(p, 3),
                  "interpretación": interpretar(b, p)})

# Efecto total en jóvenes: combinación beta1 + beta3
tj = res_final.t_test("ln_Kaitz_c + interaccion_joven = 0")
b, ee, p = float(np.squeeze(tj.effect)), float(np.squeeze(tj.sd)), float(np.squeeze(tj.pvalue))
filas.append({"variable": "β₁ + β₃", "descripción": "Efecto total del Kaitz en jóvenes",
              "coef": f"{b:.3f}{estrellas(p)}", "ee": round(ee, 3), "p_valor": round(p, 3),
              "interpretación": f"JÓVENES: si el Kaitz sube 10 %, el desempleo juvenil {signo(b)} "
                                f"{abs(b*LN10):.2f} pp{nota_sig(p)}."})

tabla_coef = pd.DataFrame(filas).set_index("variable")

print(f"Modelo principal | Dependiente: tasa de desempleo (%) | N = {int(res_final.nobs)} | "
      f"R² = {res_final.rsquared:.3f} | R² ajustado = {res_final.rsquared_adj:.3f}")
print(f"Errores estándar: {COV_ELEGIDO}  |  *** p<0.01, ** p<0.05, * p<0.10")
with pd.option_context("display.max_colwidth", None):
    display(tabla_coef)

tabla_coef.to_excel("datos_procesados/tabla_coeficientes_modelo_principal.xlsx")
print("Tabla guardada en 'datos_procesados/tabla_coeficientes_modelo_principal.xlsx'")

### 5.1 Tabla de resultados de las regresiones (`tabla_resultados_regresiones`)

**Qué representa:** cada **fila** es uno de los modelos estimados (el principal y las pruebas de robustez) y las **columnas** muestran cuánto afecta el salario mínimo (Kaitz) a la variable dependiente en ese modelo. Sirve para ver de un vistazo si el resultado se mantiene cuando se cambia algo del modelo. Los modelos 2 y 3 (por grupo) no aparecen aquí porque no tienen interacción; sus resultados están en la sección 2.

#### Qué significa cada columna

| Columna | Qué es |
|---|---|
| `N` | Número de observaciones usadas en ese modelo |
| `R2_aj` | R² ajustado: qué parte de la variación de la dependiente explica el modelo (0,97 = 97 %) |
| `β_adultos` | **β₁**: coeficiente del Kaitz para los adultos |
| `(ee_adultos)` | Error estándar de β₁: su margen de incertidumbre |
| `pp+10%_adultos` | **Cuántos puntos porcentuales cambia el desempleo adulto si el Kaitz sube 10 %** |
| `β_jóvenes`, `(ee_jóvenes)`, `pp+10%_jóvenes` | Lo mismo para los jóvenes (**β₁ + β₃**) |
| `β_dif`, `(ee_dif)`, `pp+10%_dif` | **β₃**: cuánto más afecta a los jóvenes que a los adultos |
| `p_unil_H2(β3>0)` | p-valor de una cola para H2. Si es menor que 0,05, el efecto es mayor en jóvenes |

Las **estrellas** indican significancia: \*\*\* al 1 %, \*\* al 5 % y \* al 10 %. Si un coeficiente no tiene estrellas, no se puede afirmar que sea distinto de cero.

**Ojo con dos filas:**
- En **(2) Ocupación** la dependiente es la tasa de ocupación, así que se esperan coeficientes **negativos**, y H2 se lee al revés (β₃ < 0). Por eso ahí `p_unil_H2` sale cercano a 1.
- En **(3) Salario mínimo real**, `pp+10%` se refiere a un alza de 10 % del salario real, no del Kaitz.

In [ ]:
# ============================================================
# 5.1 TABLA DE RESULTADOS DE LAS REGRESIONES (tabla_resultados_regresiones)
# ============================================================
def estrellas(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

especificaciones = {
    "(1) Principal":               (res_final,   "ln_Kaitz_c",        "interaccion_joven"),
    "(2) Dep.: tasa de ocupación": (res_ocup,    "ln_Kaitz_c",        "interaccion_joven"),
    "(3) Salario mínimo real":     (res_sr,      "ln_salario_real_c", "interaccion_joven_sr"),
    "(4) Kaitz rezagado 1 trim.":  (res_lag[1],  "ln_Kaitz_c_L1",     "interaccion_joven_L1"),
    "(5) Kaitz rezagado 2 trim.":  (res_lag[2],  "ln_Kaitz_c_L2",     "interaccion_joven_L2"),
    "(6) Sin periodo COVID":       (res_nocovid, "ln_Kaitz_c",        "interaccion_joven"),
    "(7) Con tendencia lineal":    (res_tend,    "ln_Kaitz_c",        "interaccion_joven"),
}

filas = []
for nombre, (r, var, inter) in especificaciones.items():
    t = tabla_efectos(r, var, inter)
    fila = {"especificación": nombre, "N": int(r.nobs), "R2_aj": r.rsquared_adj}
    for etiqueta, clave in [("adultos", "β1: adultos"), ("jóvenes", "β1+β3: jóvenes"),
                            ("dif", "β3: diferencia jóvenes-adultos (H2)")]:
        fila[f"β_{etiqueta}"] = f"{t.loc[clave, 'beta']:.3f}{estrellas(t.loc[clave, 'p_bilateral'])}"
        fila[f"(ee_{etiqueta})"] = f"({t.loc[clave, 'ee']:.3f})"
        fila[f"pp+10%_{etiqueta}"] = t.loc[clave, "pp_ante_+10%"]
    fila["p_unil_H2(β3>0)"] = t.loc["β3: diferencia jóvenes-adultos (H2)", "p_unilateral(beta>0)"]
    filas.append(fila)

tabla_resultados_regresiones = pd.DataFrame(filas).set_index("especificación")
print(f"Errores estándar: {COV_ELEGIDO}  |  *** p<0.01, ** p<0.05, * p<0.10 (bilaterales)")
print("pp+10% = puntos porcentuales ante un alza de 10 % del Kaitz (o del salario real).")
print("En (2) la dependiente es la tasa de ocupación: el signo esperado es negativo y la H2 se lee al revés (β3<0).")
display(tabla_resultados_regresiones)

# Exportar para pegar en el informe
tabla_resultados_regresiones.to_excel("datos_procesados/tabla_resultados_regresiones.xlsx")
print("Tabla guardada en 'datos_procesados/tabla_resultados_regresiones.xlsx'")

#### Cómo leer una fila: ejemplo con **(1) Principal**

Se toman las columnas `pp+10%`, porque los β "crudos" son difíciles de leer directamente al estar el Kaitz en logaritmo:
- `pp+10%_adultos`: si el Kaitz sube **10 %**, el desempleo de los **adultos** sube esa cantidad de puntos porcentuales (por ejemplo, 0,98 pp lleva el desempleo de 6,5 % a 7,5 %).
- `pp+10%_jóvenes`: lo mismo para los **jóvenes**.
- `pp+10%_dif`: cuántos pp **más** sube el desempleo juvenil que el adulto. Si `p_unil_H2` es menor que 0,05, esa diferencia es significativa y **se cumple H2**.

#### Cómo usar la tabla completa

Hay que recorrer las filas y preguntarse: **¿los signos y las estrellas se mantienen en todas?**

| Fila | Qué cambia respecto del modelo principal | Qué se espera si el resultado es robusto |
|---|---|---|
| (2) Ocupación | Dependiente = tasa de ocupación | Coeficientes **negativos** (la ocupación cae) |
| (3) Salario real | Otra medida de salario mínimo | Coeficientes positivos y significativos |
| (4) y (5) Rezagos | Kaitz de 1 y 2 trimestres atrás | Magnitudes parecidas al principal |
| (6) Sin COVID | Se excluye la pandemia | Magnitudes parecidas al principal |
| (7) Tendencia | Se controla por una tendencia lineal | Coeficientes positivos y significativos |

Si en todas las filas el efecto mantiene el signo esperado y la significancia, se dice que el resultado es **robusto**. En el informe conviene comentar sobre todo las columnas `pp+10%`.

### 5.2 Conclusión sobre las hipótesis
Esta celda traduce los números del modelo principal (con errores robustos) a frases que puedes adaptar para la Etapa 5:
- **H1** usa el p-valor unilateral de β₁ (adultos) y de β₁+β₃ (jóvenes).
- **H2** usa el p-valor unilateral de β₃.
- La última parte compara el efecto en desempleo con el efecto en ocupación para los jóvenes. Si la ocupación cae más de lo que sube el desempleo, la diferencia corresponde a jóvenes que **salieron de la fuerza laboral**. Esta es la interpretación que pide la sección 4.5 de la pauta.

In [ ]:
# ============================================================
# 5.2 LECTURA AUTOMÁTICA DE H1 Y H2 (modelo principal)
# ============================================================
t = tabla_efectos(res_final)
b1, b13, b3 = t["beta"]
p_h1_adu = t.loc["β1: adultos", "p_unilateral(beta>0)"]
p_h1_jov = t.loc["β1+β3: jóvenes", "p_unilateral(beta>0)"]
p_h2 = t.loc["β3: diferencia jóvenes-adultos (H2)", "p_unilateral(beta>0)"]

print("H1 (el salario mínimo aumenta el desempleo):")
print(f"  Adultos: un alza de 10% del Kaitz cambia el desempleo en {b1*np.log(1.1):+.2f} pp "
      f"(p unilateral = {p_h1_adu:.3f}) -> {'se rechaza H0: evidencia a favor de H1' if p_h1_adu < ALPHA else 'no se rechaza H0'}")
print(f"  Jóvenes: un alza de 10% del Kaitz cambia el desempleo en {b13*np.log(1.1):+.2f} pp "
      f"(p unilateral = {p_h1_jov:.3f}) -> {'se rechaza H0: evidencia a favor de H1' if p_h1_jov < ALPHA else 'no se rechaza H0'}")
print("\nH2 (el efecto es mayor en jóvenes, β3 > 0):")
print(f"  β3 = {b3:.3f}; diferencia ante +10% = {b3*np.log(1.1):+.2f} pp "
      f"(p unilateral = {p_h2:.3f}) -> {'se rechaza H0: evidencia a favor de H2' if p_h2 < ALPHA else 'no se rechaza H0'}")

t_oc = tabla_efectos(res_ocup)
print("\nCanal de salida de la fuerza laboral (comparar efectos absolutos en pp, jóvenes):")
print(f"  Desempleo: {b13*np.log(1.1):+.2f} pp | Ocupación: {t_oc.loc['β1+β3: jóvenes','beta']*np.log(1.1):+.2f} pp")
print("  Si la caída de la ocupación es mayor que el alza del desempleo, parte de los jóvenes "
      "estaría saliendo de la fuerza laboral (Brown et al., 1983).")